In [0]:
from pyspark.sql import functions as F

def verificar_completude(nome_tabela):
    df = spark.table(f"workspace.mvp_gramados.{nome_tabela}")

    expressoes = [F.count("*").alias("total")]

    for i, coluna in enumerate(df.columns):
        valor = F.col(coluna)
        ausente = (
            valor.isNull()
            | (F.length(F.trim(valor.cast("string"))) == 0)
        )

        expressoes.append(
            F.count(F.when(ausente, 1)).alias(f"ausentes_{i}")
        )

    resumo = df.agg(*expressoes).first()
    total = resumo["total"]

    return [
        (
            nome_tabela,
            coluna,
            total,
            resumo[f"ausentes_{i}"],
            round(resumo[f"ausentes_{i}"] / total * 100, 2)
            if total > 0 else None
        )
        for i, coluna in enumerate(df.columns)
    ]

registros = (
    verificar_completude("bronze_partidas")
    + verificar_completude("bronze_gramados")
)

qualidade_completude = spark.createDataFrame(
    registros,
    """
    tabela STRING,
    coluna STRING,
    total_registros LONG,
    valores_ausentes LONG,
    percentual_ausentes DOUBLE
    """
)

display(
    qualidade_completude.orderBy(
        "tabela",
        F.desc("percentual_ausentes"),
        "coluna"
    )
)

In [0]:
partidas_brutas = spark.table(
    "workspace.mvp_gramados.bronze_partidas"
)

gramados_brutos = spark.table(
    "workspace.mvp_gramados.bronze_gramados"
)

print("IDs repetidos nas partidas:")
display(
    partidas_brutas
    .groupBy("ID")
    .count()
    .filter(F.col("count") > 1)
)

print("Nomes de estádio repetidos na pesquisa:")
display(
    gramados_brutos
    .groupBy("arena")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Linhas inteiramente duplicadas em partidas:",
    partidas_brutas.count() - partidas_brutas.dropDuplicates().count()
)

print(
    "Linhas inteiramente duplicadas em gramados:",
    gramados_brutos.count() - gramados_brutos.dropDuplicates().count()
)

In [0]:
campos_numericos = {
    "ID": 1,
    "rodata": 1,
    "mandante_Placar": 0,
    "visitante_Placar": 0
}

resultados = []

for coluna, minimo_permitido in campos_numericos.items():
    valor = F.expr(f"try_cast(`{coluna}` AS INT)")

    resumo = partidas_brutas.agg(
        F.count(F.when(valor.isNull(), 1)).alias("sem_inteiro_valido"),
        F.count(
            F.when(valor < minimo_permitido, 1)
        ).alias("abaixo_minimo"),
        F.min(valor).alias("minimo_observado"),
        F.max(valor).alias("maximo_observado")
    ).first()

    resultados.append((
        coluna,
        minimo_permitido,
        resumo["sem_inteiro_valido"],
        resumo["abaixo_minimo"],
        resumo["minimo_observado"],
        resumo["maximo_observado"]
    ))

display(spark.createDataFrame(
    resultados,
    """
    coluna STRING,
    minimo_permitido INT,
    sem_inteiro_valido LONG,
    abaixo_minimo LONG,
    minimo_observado INT,
    maximo_observado INT
    """
))

In [0]:
data_convertida = F.expr(
    "cast(try_to_timestamp(data, 'dd/MM/yyyy') AS DATE)"
)

hora_valida = F.col("hora").rlike(
    r"^([01][0-9]|2[0-3]):[0-5][0-9](:[0-5][0-9])?$"
)

display(
    partidas_brutas.agg(
        F.count(
            F.when(data_convertida.isNull(), 1)
        ).alias("datas_ausentes_ou_invalidas"),

        F.count(
            F.when(
                F.col("hora").isNull() | (~hora_valida),
                1
            )
        ).alias("horarios_ausentes_ou_invalidos"),

        F.min(data_convertida).alias("primeira_data"),
        F.max(data_convertida).alias("ultima_data")
    )
)

print("Registros com problema de data ou horário:")

display(
    partidas_brutas
    .filter(
        data_convertida.isNull()
        | F.col("hora").isNull()
        | (~hora_valida)
    )
    .select("ID", "data", "hora", "mandante", "visitante")
)

In [0]:
conferencia_resultados = (
    partidas_brutas
    .withColumn(
        "gols_mandante",
        F.expr("try_cast(mandante_Placar AS INT)")
    )
    .withColumn(
        "gols_visitante",
        F.expr("try_cast(visitante_Placar AS INT)")
    )
    .withColumn(
        "vencedor_calculado",
        F.when(
            F.col("gols_mandante") > F.col("gols_visitante"),
            F.col("mandante")
        )
        .when(
            F.col("gols_mandante") < F.col("gols_visitante"),
            F.col("visitante")
        )
        .otherwise(F.lit("-"))
    )
)

divergencias = conferencia_resultados.filter(
    ~F.col("vencedor").eqNullSafe(F.col("vencedor_calculado"))
)

print("Partidas com vencedor diferente do placar:", divergencias.count())

display(
    divergencias.select(
        "ID", "data", "mandante", "visitante",
        "mandante_Placar", "visitante_Placar",
        "vencedor", "vencedor_calculado"
    )
)

In [0]:
ufs_validas = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO"
]

problemas_estados = partidas_brutas.filter(
    F.col("mandante_Estado").isNull()
    | (~F.col("mandante_Estado").isin(ufs_validas))
    | F.col("visitante_Estado").isNull()
    | (~F.col("visitante_Estado").isin(ufs_validas))
)

mesmo_clube = partidas_brutas.filter(
    F.lower(F.trim(F.col("mandante")))
    == F.lower(F.trim(F.col("visitante")))
)

print("Partidas com sigla de estado inválida:", problemas_estados.count())

display(
    problemas_estados.select(
        "ID", "mandante", "mandante_Estado",
        "visitante", "visitante_Estado"
    )
)

print("Partidas com mandante igual ao visitante:", mesmo_clube.count())

display(
    mesmo_clube.select("ID", "data", "mandante", "visitante")
)

In [0]:
tipos_validos = [
    "natural", "sintetico", "hibrido", "nao_confirmado"
]

conferencia_gramados = (
    gramados_brutos
    .withColumn(
        "inicio_data",
        F.expr("try_cast(inicio_validade AS DATE)")
    )
    .withColumn(
        "fim_data",
        F.expr("try_cast(fim_validade AS DATE)")
    )
)

def preenchido(coluna):
    return (
        F.col(coluna).isNotNull()
        & (F.length(F.trim(F.col(coluna))) > 0)
    )

regras = {
    "tipo_ausente_ou_invalido":
        F.col("tipo_gramado").isNull()
        | (~F.col("tipo_gramado").isin(tipos_validos)),

    "inicio_preenchido_mas_invalido":
        preenchido("inicio_validade")
        & F.col("inicio_data").isNull(),

    "fim_preenchido_mas_invalido":
        preenchido("fim_validade")
        & F.col("fim_data").isNull(),

    "classificado_sem_periodo_valido":
        F.col("tipo_gramado").isin("natural", "sintetico", "hibrido")
        & (
            F.col("inicio_data").isNull()
            | F.col("fim_data").isNull()
        ),

    "periodo_invertido":
        F.col("inicio_data") > F.col("fim_data")
}

display(
    conferencia_gramados.agg(*[
        F.count(F.when(condicao, 1)).alias(nome)
        for nome, condicao in regras.items()
    ])
)

display(
    conferencia_gramados
    .groupBy("tipo_gramado")
    .count()
    .orderBy("tipo_gramado")
)

In [0]:
jogos_gols = (
    spark.table("workspace.mvp_gramados.silver_partidas_gramados")
    .withColumn(
        "total_gols",
        F.col("mandante_Placar") + F.col("visitante_Placar")
    )
)

# Distribuição: quantas partidas tiveram cada total de gols?
display(
    jogos_gols
    .groupBy("total_gols")
    .count()
    .orderBy("total_gols")
)

# Dez partidas com os maiores totais de gols
display(
    jogos_gols
    .select(
        "ID", "data", "mandante", "visitante",
        "mandante_Placar", "visitante_Placar",
        "total_gols", "tipo_gramado", "status_gramado"
    )
    .orderBy(F.desc("total_gols"), "ID")
    .limit(10)
)

Inspeção de valores extremos — gols

Nas 760 partidas de 2023–2024, o total de gols variou de 0 a 10. Foram identificadas quatro partidas com oito gols e uma com dez gols. Esses valores foram mantidos, pois sua magnitude, isoladamente, não comprova erro de registro. Esta inspeção descreve a distribuição e não substitui a conferência dos placares em fontes externas. A análise por tipo de gramado considera somente as 737 partidas com classificação válida para a data do jogo; as outras 23 permanecem na base, mas são excluídas dos agregados analíticos.

In [0]:
base = spark.table("workspace.mvp_gramados.silver_partidas_gramados")

display(
    base.agg(
        F.count("*").alias("total_partidas"),
        F.countDistinct("ID").alias("ids_distintos")
    )
)

display(
    base.groupBy("status_gramado")
    .count()
    .orderBy("status_gramado")
)

display(
    base.filter(F.col("status_gramado") != "dentro_do_periodo")
    .groupBy("arena", "status_gramado")
    .count()
    .orderBy(F.desc("count"), "arena")
)